# 06 — MCMC: Gibbs Sampling & Hamiltonian Monte Carlo
**References:** Geman & Geman (1984) · Neal (2011) *MCMC using HMC* · Betancourt (2017) *Conceptual Intro to HMC*

## Narrative thread
```
Gibbs: full conditionals -> block Gibbs -> data augmentation -> HMC: Hamiltonian dynamics -> NUTS
```

## 1. Gibbs sampling

**Idea:** Instead of proposing a joint move for all parameters, sample each component from its **full conditional distribution** — the posterior of one parameter given all others and the data.

**Algorithm:** For $\boldsymbol{\theta} = (\theta_1, \ldots, \theta_p)$, at each iteration cycle:
$$\theta_1^{(t+1)} \sim p(\theta_1 \mid \theta_2^{(t)}, \ldots, \theta_p^{(t)}, \mathbf{x})$$
$$\theta_2^{(t+1)} \sim p(\theta_2 \mid \theta_1^{(t+1)}, \theta_3^{(t)}, \ldots, \theta_p^{(t)}, \mathbf{x})$$
$$\vdots$$
$$\theta_p^{(t+1)} \sim p(\theta_p \mid \theta_1^{(t+1)}, \ldots, \theta_{p-1}^{(t+1)}, \mathbf{x})$$

Gibbs sampling is a special case of MH with acceptance probability exactly 1 — every proposal is accepted.

**When it works well:** When full conditionals are known distributions (e.g., in conjugate hierarchical models).

**Weakness:** High correlation between parameters → slow mixing (the chain moves in zigzags along the correlation direction).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Gibbs sampler for Normal-Normal model ─────────────────────────────────
# X1,...,Xn | mu ~ N(mu, sigma^2)  -- sigma known
# mu | tau^2 ~ N(mu0, tau^2)       -- conjugate prior
# Full conditionals:
#   mu | X,sigma^2,tau^2 ~ N(mu_n, tau_n^2)
#   (Only 1 parameter here so we'll demo on bivariate Normal)

# Gibbs for bivariate Normal with known rho
# Full conditionals:
#   X | Y=y ~ N(rho*y, 1-rho^2)
#   Y | X=x ~ N(rho*x, 1-rho^2)

def gibbs_bivariate_normal(rho, n_iter, warmup=1000):
    x, y = 0.0, 0.0
    chain_x, chain_y = [x], [y]
    cond_var = 1 - rho**2
    for _ in range(n_iter - 1):
        x = np.random.normal(rho * y, np.sqrt(cond_var))
        y = np.random.normal(rho * x, np.sqrt(cond_var))
        chain_x.append(x); chain_y.append(y)
    return np.array(chain_x[warmup:]), np.array(chain_y[warmup:])

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
rhos = [0.0, 0.7, 0.97]
n_iter_gibbs = 5000; warmup_g = 500

for col, rho in enumerate(rhos):
    cx, cy = gibbs_bivariate_normal(rho, n_iter_gibbs, warmup_g)

    # Scatter of samples vs true contour
    ax0 = axes[0, col]
    ax0.scatter(cx[:500], cy[:500], s=4, alpha=0.3, color='#4361ee')
    # True bivariate normal contour
    x_g = np.linspace(-3.5, 3.5, 100); y_g = np.linspace(-3.5, 3.5, 100)
    XX, YY = np.meshgrid(x_g, y_g)
    Sigma = np.array([[1, rho], [rho, 1]])
    ZZ = stats.multivariate_normal.pdf(np.dstack([XX, YY]), mean=[0,0], cov=Sigma)
    ax0.contour(XX, YY, ZZ, levels=5, colors='#f72585', linewidths=1.5)
    ax0.set_title(f'$\\rho={rho}$ — samples vs true contours')
    ax0.set_xlabel('x'); ax0.set_ylabel('y')

    # Trace plot of x showing autocorrelation
    ax1 = axes[1, col]
    ax1.plot(cx[:300], cy[:300], lw=0.5, alpha=0.7, color='#4361ee')
    ax1.set_xlabel('x'); ax1.set_ylabel('y')
    ess_x = len(cx) / (1 + 2*sum(np.corrcoef(cx[:-k], cx[k:])[0,1] for k in range(1, 50) if np.corrcoef(cx[:-k], cx[k:])[0,1] > 0.05))
    ax1.set_title(f'$\\rho={rho}$: trajectory (300 steps)\nESS≈{ess_x:.0f}/{len(cx)} — higher ρ → worse mixing')

plt.suptitle('Gibbs Sampling for Bivariate Normal\nHigh correlation causes slow mixing (zigzag trajectory)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Hamiltonian Monte Carlo

**Idea:** Augment $\theta$ with a momentum variable $\mathbf{r}$ and simulate Hamiltonian dynamics to make distant, low-autocorrelation proposals.

**The Hamiltonian:**
$$H(\theta, \mathbf{r}) = \underbrace{-\log p(\theta \mid \mathbf{x})}_{\text{potential energy } U(\theta)} + \underbrace{\frac{1}{2}\mathbf{r}^\top M^{-1} \mathbf{r}}_{\text{kinetic energy } K(\mathbf{r})}$$

**Hamilton's equations:**
$$\frac{d\theta}{dt} = \frac{\partial H}{\partial \mathbf{r}} = M^{-1}\mathbf{r}, \qquad \frac{d\mathbf{r}}{dt} = -\frac{\partial H}{\partial \theta} = \nabla_\theta \log p(\theta \mid \mathbf{x})$$

**Leapfrog integrator** (symplectic, reversible, volume-preserving):
1. Half step for momentum: $\mathbf{r}_{t+\varepsilon/2} = \mathbf{r}_t + \frac{\varepsilon}{2}\nabla_\theta \log p(\theta_t)$
2. Full step for position: $\theta_{t+\varepsilon} = \theta_t + \varepsilon M^{-1}\mathbf{r}_{t+\varepsilon/2}$
3. Half step for momentum: $\mathbf{r}_{t+\varepsilon} = \mathbf{r}_{t+\varepsilon/2} + \frac{\varepsilon}{2}\nabla_\theta \log p(\theta_{t+\varepsilon})$

After $L$ leapfrog steps, apply MH correction to account for discretization error. HMC achieves acceptance rates near 65–90% even in high dimensions.

**NUTS (No-U-Turn Sampler):** Hoffman & Gelman (2014) — automatically chooses $L$ and $\varepsilon$. Powers Stan, PyMC, NumPyro.

In [ ]:
# ── HMC implementation from scratch for 2D Normal ─────────────────────────
def hmc_2d(log_prob_and_grad, n_iter, step_size, n_leapfrog, theta_init):
    """
    HMC for 2D target. log_prob_and_grad returns (log_prob, grad).
    """
    theta   = np.array(theta_init, dtype=float)
    chain   = [theta.copy()]
    n_acc   = 0

    for _ in range(n_iter - 1):
        r = np.random.normal(0, 1, 2)       # fresh momentum
        theta_prop = theta.copy()
        r_prop     = r.copy()
        lp_curr, _  = log_prob_and_grad(theta)
        H_curr      = -lp_curr + 0.5 * np.dot(r, r)

        # Leapfrog
        _, grad = log_prob_and_grad(theta_prop)
        r_prop += 0.5 * step_size * grad
        for _ in range(n_leapfrog):
            theta_prop += step_size * r_prop
            _, grad     = log_prob_and_grad(theta_prop)
            r_prop     += step_size * grad
        _, grad = log_prob_and_grad(theta_prop)
        r_prop += 0.5 * step_size * grad

        lp_prop, _  = log_prob_and_grad(theta_prop)
        H_prop      = -lp_prop + 0.5 * np.dot(r_prop, r_prop)

        if np.log(np.random.uniform()) < (H_curr - H_prop):
            theta = theta_prop.copy()
            n_acc += 1
        chain.append(theta.copy())

    return np.array(chain), n_acc / (n_iter - 1)

# Correlated 2D Normal: compare HMC vs Gibbs
rho = 0.97
Sigma_hmc = np.array([[1, rho], [rho, 1]])
Sigma_inv  = np.linalg.inv(Sigma_hmc)

def log_prob_grad(theta):
    lp   = -0.5 * theta @ Sigma_inv @ theta
    grad = -Sigma_inv @ theta
    return lp, grad

n_hmc   = 2000; warmup_hmc = 200
chain_hmc, accept_hmc = hmc_2d(log_prob_grad, n_hmc, step_size=0.3, n_leapfrog=20, theta_init=[0,0])
cx_gibbs, cy_gibbs = gibbs_bivariate_normal(rho, n_iter=n_hmc, warmup=warmup_hmc)

hmc_post = chain_hmc[warmup_hmc:]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

x_g = np.linspace(-3.5, 3.5, 100); y_g = np.linspace(-3.5, 3.5, 100)
XX, YY = np.meshgrid(x_g, y_g)
ZZ     = stats.multivariate_normal.pdf(np.dstack([XX,YY]), mean=[0,0], cov=Sigma_hmc)

for ax, (sx, sy, name, col) in zip(axes[:2], [
    (cx_gibbs, cy_gibbs, f'Gibbs (ρ={rho})', '#4361ee'),
    (hmc_post[:,0], hmc_post[:,1], f'HMC (ρ={rho})', '#f72585'),
]):
    ax.scatter(sx[:300], sy[:300], s=5, alpha=0.4, color=col)
    ax.contour(XX, YY, ZZ, levels=5, colors='gray', linewidths=1, alpha=0.6)
    ax.set_xlabel('$\\theta_1$'); ax.set_ylabel('$\\theta_2$')
    ess = len(sx) / (1 + 2*abs(sum(np.corrcoef(sx[:-k], sx[k:])[0,1] for k in range(1,30) if abs(np.corrcoef(sx[:-k], sx[k:])[0,1]) > 0.05)))
    ax.set_title(f'{name}\nESS≈{ess:.0f} / {len(sx)} iters')

# Autocorrelation comparison
lags = np.arange(0, 50)
acf_gibbs = [np.corrcoef(cx_gibbs[:-k] if k>0 else cx_gibbs, cx_gibbs[k:] if k>0 else cx_gibbs)[0,1] for k in lags]
acf_hmc   = [np.corrcoef(hmc_post[:-k,0] if k>0 else hmc_post[:,0], hmc_post[k:,0] if k>0 else hmc_post[:,0])[0,1] for k in lags]

axes[2].plot(lags, acf_gibbs, color='#4361ee', lw=2, label='Gibbs')
axes[2].plot(lags, acf_hmc,   color='#f72585', lw=2, label='HMC')
axes[2].axhline(0.05, color='gray', lw=1, linestyle='--')
axes[2].set_xlabel('Lag'); axes[2].set_ylabel('Autocorrelation')
axes[2].set_title(f'Autocorrelation: Gibbs vs HMC (ρ={rho})\nHMC decorrelates in far fewer steps')
axes[2].legend()

plt.suptitle(f'Gibbs vs HMC for correlated target (ρ={rho})\nHMC accept rate={accept_hmc:.2f}',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. HMC vs MH vs Gibbs: comparison

| Algorithm | Requires gradient? | Scales to high $d$? | Best for |
|---|---|---|---|
| Random walk MH | No | Poorly | Low-$d$, any target |
| Gibbs | No (full conditionals) | Moderate | Conjugate hierarchical models |
| HMC | Yes | Yes — $O(d^{1/4})$ | Continuous parameters, smooth posteriors |
| NUTS | Yes (auto-tuned) | Yes | Default in Stan/PyMC |

**Why HMC wins:** Random walk explores $O(\varepsilon)$ per step; HMC explores $O(L\varepsilon)$ with $L$ leapfrog steps. The gradient guides the sampler to stay on the typical set of the posterior.

## 4. Key takeaways

| Concept | Statement |
|---|---|
| Gibbs | Sample each parameter from its full conditional |
| Gibbs weakness | Slow mixing when parameters are correlated |
| HMC | Uses gradient of log-posterior to propose distant moves |
| Leapfrog | Symplectic integrator preserves volume and reversibility |
| NUTS | Auto-tunes step size and trajectory length — powers Stan |

**Next:** notebook 07 — Bayesian model checking: does the model fit the data?